In [0]:
%sql
SELECT count(*) as nb_lignes,
min(timestamp) as debut,
max(timestamp) as fin
from workspace.energy.bronze_public_power;


SELECT * 
FROM workspace.energy.bronze_public_power
ORDER BY `timestamp`
limit 10



In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Europe/Brussels")

TABLE = "workspace.energy.bronze_public_power"

last_day = spark.table(TABLE).select(F.max(F.to_date("timestamp"))).first()[0]
LOOKBACK_DAYS = 7
start = last_day - timedelta(days=LOOKBACK_DAYS)
end = date.today() - timedelta(days=1)

print("Last day in bronze:", last_day)
print("Days to fetch     :", start, "->", end)
print("Number of days    :", (end - start).days + 1)

In [0]:
import time
import requests
import pandas as pd

API_URL = "https://api.energy-charts.info/v2/public_power"


def get_history(start, end, country="be"):
    for attempt in range(3):
        r = requests.get(API_URL, params={"country": country, "start": start, "end": end}, timeout=60)
        if r.status_code == 200:
            break
        print(f"HTTP {r.status_code}, retrying in 10 s...")
        time.sleep(10)
    else:
        raise Exception(f"API failed for {start} -> {end}: HTTP {r.status_code}")

    df = pd.json_normalize(r.json()["data"])
    df.columns = [c.replace("values.", "") for c in df.columns]
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df


chunks = []
current = start
while current <= end:
    next_month = (current.replace(day=1) + timedelta(days=32)).replace(day=1)
    chunk_end = min(end, next_month - timedelta(days=1))
    print("Fetching", current, "->", chunk_end)
    chunks.append(get_history(current.isoformat(), chunk_end.isoformat()))
    current = chunk_end + timedelta(days=1)
    time.sleep(2)

new_data = pd.concat(chunks, ignore_index=True)
print(new_data.shape)
print("First:", new_data["timestamp"].min())
print("Last :", new_data["timestamp"].max())

In [0]:
updates = spark.createDataFrame(new_data)
updates = updates.withColumn("ingested_at", F.current_timestamp())
updates = updates.withColumn("source", F.lit("energy-charts/v2/public_power"))

target_schema = spark.table(TABLE).schema
for field in target_schema:
    if field.name in updates.columns:
        updates = updates.withColumn(field.name, F.col(field.name).cast(field.dataType))
    else:
        updates = updates.withColumn(field.name, F.lit(None).cast(field.dataType))
updates = updates.select([field.name for field in target_schema])

updates.createOrReplaceTempView("updates")

spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING updates AS s
    ON t.`timestamp` = s.`timestamp`
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("Merge done ✅")

In [0]:
%sql
SELECT COUNT(*) AS nb_rows,
       MAX(`timestamp`) AS last_hour
FROM workspace.energy.bronze_public_power;

In [0]:
bronze = spark.table(TABLE)

duplicates = bronze.groupBy("timestamp").count().filter("count > 1").count()

stats = bronze.agg(
    F.min(F.unix_timestamp("timestamp")).alias("first_s"),
    F.max(F.unix_timestamp("timestamp")).alias("last_s"),
    F.count("*").alias("rows"),
).first()

expected_rows = (stats["last_s"] - stats["first_s"]) // 3600 + 1

if duplicates > 0 or stats["rows"] != expected_rows:
    raise Exception(
        f"Bronze check failed: {duplicates} duplicate hours, "
        f"{stats['rows']} rows vs {expected_rows} expected"
    )

print(f"Bronze OK ✅ {stats['rows']} rows, no duplicates, no gaps")

In [0]:
%sql
SET TIME ZONE 'Europe/Brussels';

SELECT DATE_TRUNC('day', `timestamp`) AS day,
       ROUND(AVG(nuclear), 0) AS avg_nuclear_mw,
       ROUND(MAX(nuclear), 0) AS max_nuclear_mw
FROM workspace.energy.bronze_public_power
WHERE `timestamp` >= '2026-10-01'
GROUP BY 1
ORDER BY 1;